# PDAC Survival Prediction — Data Preprocessing

**Project:** Reproducing and improving the Nature Cancer (2024) PDAC Molecular Twin survival model  
**Dataset:** TCGA-PAAD from cBioPortal  
**Author:** Kumar Mahat | Texas Tech University  

---

This notebook covers:
- Loading multi-omic data from cBioPortal
- Fixing ID format mismatches (GDC UUIDs vs TCGA barcodes)
- Merging clinical and molecular data
- Initial data cleaning and validation

In [ ]:
# Install required packages
!pip install pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn requests -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print('Libraries loaded successfully')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# ============================================================
# LOAD CLINICAL DATA FROM cBioPortal
# Dataset: TCGA Pancreatic Adenocarcinoma (TCGA-PAAD)
# ============================================================

CBIO_BASE = 'https://www.cbioportal.org/api'
STUDY_ID = 'paad_tcga'

def fetch_clinical_data(study_id):
    """Fetch clinical data from cBioPortal API."""
    url = f'{CBIO_BASE}/studies/{study_id}/clinical-data'
    params = {'clinicalDataType': 'PATIENT', 'projection': 'DETAILED'}
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        df = pd.DataFrame(data)
        return df
    else:
        print(f'Error: {response.status_code}')
        return None

print('Fetching clinical data...')
clinical_raw = fetch_clinical_data(STUDY_ID)
print(f'Clinical records fetched: {len(clinical_raw) if clinical_raw is not None else 0}')

In [ ]:
# ============================================================
# PIVOT CLINICAL DATA INTO WIDE FORMAT
# ============================================================

def pivot_clinical(df):
    """Pivot clinical data from long to wide format."""
    df_pivot = df.pivot_table(
        index='patientId',
        columns='clinicalAttributeId',
        values='value',
        aggfunc='first'
    ).reset_index()
    df_pivot.columns.name = None
    return df_pivot

clinical_df = pivot_clinical(clinical_raw)
print(f'Clinical dataframe shape: {clinical_df.shape}')
print(f'Columns: {list(clinical_df.columns[:10])}...')
clinical_df.head()

In [ ]:
# ============================================================
# CREATE SURVIVAL LABELS
# Binary classification: Short-term (<12 months) vs Long-term (>=12 months)
# ============================================================

def create_survival_labels(df, survival_col='OS_MONTHS', threshold=12):
    """Create binary survival labels based on overall survival months."""
    df = df.copy()
    
    # Convert to numeric
    if survival_col in df.columns:
        df[survival_col] = pd.to_numeric(df[survival_col], errors='coerce')
        df['survival_label'] = (df[survival_col] >= threshold).astype(int)
        
        # Drop rows with missing survival info
        df = df.dropna(subset=[survival_col])
        
        print(f'Survival threshold: {threshold} months')
        print(f'Short-term survivors (<{threshold}mo): {(df["survival_label"]==0).sum()}')
        print(f'Long-term survivors (>={threshold}mo): {(df["survival_label"]==1).sum()}')
    else:
        print(f'Column {survival_col} not found. Available:', list(df.columns))
    
    return df

clinical_df = create_survival_labels(clinical_df)
print(f'\nFinal clinical shape: {clinical_df.shape}')

In [ ]:
# ============================================================
# FIX ID FORMAT MISMATCH
# Problem: GDC uses UUIDs, TCGA barcodes use different format
# Solution: Standardize to TCGA barcode format
# ============================================================

def standardize_patient_id(patient_id):
    """
    Standardize patient IDs to TCGA barcode format.
    TCGA barcodes: TCGA-XX-XXXX
    GDC UUIDs: xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx
    """
    if isinstance(patient_id, str):
        # Already in TCGA format
        if patient_id.startswith('TCGA-'):
            return patient_id[:12]  # Keep first 12 chars: TCGA-XX-XXXX
        # UUID format - would need GDC API lookup in practice
        return patient_id
    return patient_id

# Apply standardization
if 'patientId' in clinical_df.columns:
    clinical_df['patient_id_std'] = clinical_df['patientId'].apply(standardize_patient_id)
    print('Patient ID standardization complete')
    print(f'Sample IDs: {list(clinical_df["patient_id_std"].head(5))}')

In [ ]:
# ============================================================
# DATA QUALITY CHECKS
# ============================================================

def data_quality_report(df, name='Dataset'):
    """Generate a data quality report."""
    print(f'\n=== {name} Quality Report ===')
    print(f'Shape: {df.shape}')
    print(f'\nMissing values per column:')
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
    print(missing_df[missing_df['Missing Count'] > 0].head(10))
    print(f'\nDuplicate rows: {df.duplicated().sum()}')
    return missing_df

missing_report = data_quality_report(clinical_df, 'Clinical Data')

In [ ]:
# ============================================================
# SURVIVAL DISTRIBUTION VISUALIZATION
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall survival distribution
if 'OS_MONTHS' in clinical_df.columns:
    axes[0].hist(clinical_df['OS_MONTHS'].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[0].axvline(x=12, color='red', linestyle='--', label='12-month threshold')
    axes[0].set_xlabel('Overall Survival (Months)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('TCGA-PAAD: Overall Survival Distribution')
    axes[0].legend()

# Class distribution
if 'survival_label' in clinical_df.columns:
    label_counts = clinical_df['survival_label'].value_counts()
    axes[1].bar(['Short-term\n(<12 months)', 'Long-term\n(>=12 months)'],
                label_counts.values,
                color=['#e74c3c', '#2ecc71'], edgecolor='white')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Survival Label Distribution')
    for i, v in enumerate(label_counts.values):
        axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('survival_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: survival_distribution.png')

In [ ]:
# ============================================================
# SAVE PROCESSED CLINICAL DATA
# ============================================================

save_path = '/content/drive/MyDrive/PDAC_Research/processed_clinical.csv'

try:
    import os
    os.makedirs('/content/drive/MyDrive/PDAC_Research', exist_ok=True)
    clinical_df.to_csv(save_path, index=False)
    print(f'Clinical data saved to: {save_path}')
    print(f'Shape: {clinical_df.shape}')
except Exception as e:
    print(f'Could not save to Drive: {e}')
    clinical_df.to_csv('processed_clinical.csv', index=False)
    print('Saved locally: processed_clinical.csv')

print('\nPreprocessing complete!')
print(f'Patients ready for analysis: {len(clinical_df)}')